# 04 · Train the 2.5D U-Net baseline (Kaggle GPU runner)

Trains the model on the preprocessed LiTS dataset with the
[`livertumor`](https://github.com/yanishalloum/liver_tumor_classification) package.

**Settings:** *Add Input* → your private dataset **`lits-preprocessed`**;
**Internet on**; *Accelerator* → **GPU T4 x2** or **GPU P100** (one GPU is used).

1. First run interactively with `SMOKE_TEST = True` (2 short epochs, ~5 min) to catch
   errors early without spending GPU hours.
2. Then set `SMOKE_TEST = False` and use **Save Version → Save & Run All** (~1 h).

In [ ]:
SMOKE_TEST = True  # set to False for the real training run

In [ ]:
import shutil
import subprocess

REPO = "/kaggle/working/repo"
URL = "https://github.com/yanishalloum/liver_tumor_classification.git"


def run(*cmd):
    """Run a shell command and stop the notebook if it fails.

    A plain `!command` that fails does NOT stop the notebook: the next cells run and
    fail later with a misleading error. check=True raises immediately instead.
    """
    subprocess.run(cmd, check=True)


shutil.rmtree(REPO, ignore_errors=True)  # an old clone would make `git clone` fail
try:
    run("git", "clone", "-q", "--depth", "1", URL, REPO)
except subprocess.CalledProcessError as error:
    raise RuntimeError("git clone failed: is Internet turned on (Settings > Internet)?") from error
run("pip", "install", "-q", f"{REPO}[train,viz]")
run("git", "-C", REPO, "log", "--oneline", "-1")  # exact code version of this run
assert shutil.which("livertumor"), "the livertumor command was not installed"
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")

In [ ]:
from pathlib import Path

# The dataset folder is the one holding manifest.csv *and* the volumes/ directory
# (a copy of manifest.csv also sits in figures/, which must be ignored).
DATASET = next(
    p.parent for p in Path("/kaggle/input").rglob("manifest.csv") if (p.parent / "volumes").is_dir()
)
print(DATASET)

In [ ]:
args = (
    ["--epochs", "2", "--samples-per-epoch", "200", "--max-val-cases", "3"]
    if SMOKE_TEST
    else ["--epochs", "50"]
)
run(
    "livertumor",
    "train",
    "--dataset-dir",
    str(DATASET),
    "--out-dir",
    "/kaggle/working/run_unet2d",
    *args,
)

**Learning curves.** Training loss should go down; validation Dice should go up, then
level off. If validation Dice drops while the loss keeps falling, the model is
overfitting.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

m = pd.read_csv("/kaggle/working/run_unet2d/metrics.csv")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(m.epoch, m.train_loss)
ax1.set(title="Training loss", xlabel="epoch")
ax2.plot(m.epoch, m.val_dice_liver, label="liver")
ax2.plot(m.epoch, m.val_dice_tumor, label="tumor")
ax2.set(title="Validation Dice (3D, per patient)", xlabel="epoch", ylim=(0, 1))
ax2.legend()
fig.savefig("/kaggle/working/run_unet2d/curves.png", dpi=150, bbox_inches="tight")
m.tail()

**Validation report.** Dice, lesion detection and patient-level sensitivity/specificity
on the *validation* patients, plus a few prediction figures. The test set is not used
here: it is evaluated once, at the very end of the project.

In [ ]:
run(
    "livertumor",
    "evaluate",
    "--run-dir",
    "/kaggle/working/run_unet2d",
    "--dataset-dir",
    str(DATASET),
    "--part",
    "val",
)

In [ ]:
import json

from IPython.display import Image, display

EVAL = Path("/kaggle/working/run_unet2d/eval_val")
print(json.dumps(json.loads((EVAL / "summary.json").read_text()), indent=2))
for png in sorted(EVAL.glob("prediction_case_*.png")):
    display(Image(str(png)))

In [ ]:
shutil.rmtree(REPO)  # keep only the run in the saved output